In [1]:
import sys
from pathlib import Path

# Add project root to path so imports work from the notebook's location
ROOT = str(Path.cwd().parents[1]) if Path.cwd().name == "image_jepa" else str(Path.cwd())
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

#==========================================================
from dataset import get_train_transforms,get_val_transforms,ImageDataset
from main import ResNet18,ImageSSL,LARS,WarmupCosineScheduler
from eval import LinearProbe
from eb_jepa.losses import VICRegLoss

#==========================================================
import torch 
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10
import torch.nn.functional as F 

ModuleNotFoundError: No module named 'dataset'

# Data 

In [10]:
train_data =  CIFAR10(root="/tmp/cifar10",train=True,download=True)

train_ds = ImageDataset(
    dataset= train_data, 
    transform= get_train_transforms(),
    num_crops= 2 
)

val_ds =  CIFAR10(root="/tmp/cifar10",train=False,download=True,transform=get_val_transforms())


train_loader = DataLoader(train_ds,batch_size=32,shuffle=False)
val_loader   = DataLoader(val_ds,    batch_size=32,shuffle=False)

In [11]:
print(len(train_loader))
print(len(val_loader))

1563
313


In [13]:
for batch in train_loader: 
    print(len(batch))
    views,label = batch 
    view0,view1 = views
    print(view0.shape,view1.shape)
    print(label.shape)
    print(label)
    break 

2
torch.Size([32, 3, 32, 32]) torch.Size([32, 3, 32, 32])
torch.Size([32])
tensor([6, 9, 9, 4, 1, 1, 2, 7, 8, 3, 4, 7, 7, 2, 9, 9, 9, 3, 2, 6, 4, 3, 6, 6,
        2, 6, 3, 5, 4, 0, 0, 9])


In [14]:
for batch in val_loader: 
    print(len(batch ))
    x,y = batch 
    print(x.shape)
    print(y.shape)
    break 

2
torch.Size([32, 3, 32, 32])
torch.Size([32])


# Model 

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

backbone = ResNet18()

features_dim = backbone.features_dim

model = ImageSSL(
    backbone= backbone,
    features_dim = features_dim,    # 512
    proj_hidden_dim=2048,   # from cfgs/default.yaml
    proj_output_dim=2048    # from cfgs/default.yaml

)
model = model.to(device)

linear_probe = LinearProbe(feature_dim=features_dim, num_classes=10).to(device) 


# Loss,Optimizer,Scheduler

In [16]:
loss_fn = VICRegLoss(std_coeff= 1.0, cov_coeff= 80.0)
optimizer = LARS(
    [
        {"params":model.parameters(),"lr":0.3},
        {"params":linear_probe.parameters(),"lr":0.1}
    ],
    weight_decay=1.0e-4,
    eta = 0.02,
    clip_lr=True,
    exclude_bias_n_norm=True,
    momentum=0.9
)

scheduler = WarmupCosineScheduler(
    optimizer,
    warmup_epochs= 10,
    max_epochs= 300,
    base_lr=0.3,
    min_lr=0.0,
    warmup_start_lr=3.0e-5
)

# Train 

In [ ]:

for epoch in range(100):

    model.train()
    linear_probe.train()
    epoch_loss = 0 

    for views,labels in train_loader: 
        #============================== forward pass 
        view0,view1 = views
        features,z1 = model(view0)
        _,       z2 = model(view1)
        print(features.shape,z1.shape)
        #==============================
        ssl_loss = loss_fn(z1,z2)["loss"]
        probe_loss = F.cross_entropy(linear_probe(features.detach()),labels)    # use y_hat
        #==============================
        total_loss = ssl_loss + probe_loss 
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        epoch_loss += ssl_loss.item()
        print(epoch_loss)
        break 

    scheduler.step(epoch)
    avg_loss = epoch_loss / len(train_loader)
    # print(f'Epoch {epoch}: loss = {avg_loss:.4f}')

torch.Size([32, 512]) torch.Size([32, 2048])
1.6048805713653564
torch.Size([32, 512]) torch.Size([32, 2048])
1.6128456592559814
torch.Size([32, 512]) torch.Size([32, 2048])
1.60123872756958
torch.Size([32, 512]) torch.Size([32, 2048])
1.5935264825820923
torch.Size([32, 512]) torch.Size([32, 2048])
1.5881484746932983
torch.Size([32, 512]) torch.Size([32, 2048])
1.5904277563095093
torch.Size([32, 512]) torch.Size([32, 2048])
1.5889235734939575
torch.Size([32, 512]) torch.Size([32, 2048])
1.589918851852417
torch.Size([32, 512]) torch.Size([32, 2048])
1.5856772661209106
torch.Size([32, 512]) torch.Size([32, 2048])
1.5971077680587769
torch.Size([32, 512]) torch.Size([32, 2048])
1.6010671854019165
torch.Size([32, 512]) torch.Size([32, 2048])
1.5726497173309326
torch.Size([32, 512]) torch.Size([32, 2048])
1.5877894163131714
torch.Size([32, 512]) torch.Size([32, 2048])
1.577089786529541
torch.Size([32, 512]) torch.Size([32, 2048])
1.5874552726745605
torch.Size([32, 512]) torch.Size([32, 2048])

# Evaluation 

In [ ]:
from torch.amp import autocast 

def evaluate_linear_porbe(model,linear_probe,val_loader,device,use_amp=True): 
    model.eval()
    linear_probe.eval()

    total_loss = 0 
    correct = 0 
    total = 0 

    with torch.no_grad(): 
        for data,target in val_loader:
            data = data.to(device,non_blocking=True)
            target = target.to(device,non_blocking = True)

            with autocast("cuda",enabled=use_amp):
                features, _ = model(data)

            outputs = linear_probe(features.float())
            # print(outputs)
            loss = F.cross_entropy(outputs,target)

            total_loss += loss.item()
            _,predicted = outputs.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()

            break 

    accuracy = 100 * correct / total
    avg_loss = total_loss /len(val_loader)

    return accuracy,avg_loss


In [23]:
evaluate_linear_porbe(model,linear_probe,val_loader,device=device,use_amp=False)

tensor([[   2.1064,  -79.0235,    5.1196,   57.9810,   61.0904, -124.2085,
           66.0295,   46.4389,  -91.8992,   57.7250],
        [   4.6179,  -96.6060,    4.5447,   73.8175,   71.5736, -152.9698,
           78.4163,   60.2799, -114.7801,   74.0463],
        [   2.8580,  -79.4175,    5.2874,   58.8771,   60.2636, -125.3456,
           65.8160,   48.0893,  -93.3935,   58.8644],
        [   2.9127,  -78.9927,    4.8855,   58.9547,   59.4654, -125.0428,
           65.6411,   48.5743,  -93.3506,   59.1315],
        [   2.2145,  -78.2425,    4.8848,   57.5226,   60.5409, -123.4931,
           65.4690,   46.3030,  -91.2521,   57.5894],
        [   1.8423,  -76.8798,    5.2487,   57.7022,   59.8783, -121.5878,
           64.8783,   45.5404,  -90.6578,   55.1796],
        [   2.0844,  -78.8288,    5.7120,   59.9168,   60.0447, -125.2773,
           66.2264,   47.8523,  -94.0929,   57.6571],
        [   1.5333,  -77.3194,    4.8532,   58.5993,   60.2463, -122.8204,
           64.1163,   

(18.75, 0.22383644177129094)